# Notebook 1 — Preprocessing & Feature Engineering
### H&M Customer Segmentation Project

This notebook loads the raw H&M dataset, cleans it, and engineers a four-layer behavioral feature profile per customer:

- **Layer 1:** RFM (Recency, Frequency, Monetary)
- **Layer 2:** Temporal Dynamics
- **Layer 3:** Product Affinity
- **Layer 4:** Loyalty & Repeat Behavior

**Changes from v1:**
- Added `repeat_purchase_rate` feature (Layer 4)
- Normalized `purchase_trend_slope` to [-1, +1] range
- Replaced `LabelEncoder` with frequency encoding for categorical columns

Output: `HM_features.csv` saved to Google Drive

## Cell 1 — Mount Drive & Import Libraries

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

BASE = '/content/drive/MyDrive/HM_Segmentation/data/'
print('Drive mounted and libraries loaded.')

Mounted at /content/drive
Drive mounted and libraries loaded.


## Cell 2 — Load Data

In [2]:
print('Loading transactions...')
transactions = pd.read_csv(BASE + 'transactions_train.csv',
                           dtype={'article_id': str, 'customer_id': str},
                           parse_dates=['t_dat'])

print('Loading customers...')
customers = pd.read_csv(BASE + 'customers.csv',
                        dtype={'customer_id': str})

print('Loading articles...')
articles = pd.read_csv(BASE + 'articles.csv',
                       dtype={'article_id': str})

print(f'Transactions: {transactions.shape}')
print(f'Customers:    {customers.shape}')
print(f'Articles:     {articles.shape}')

Loading transactions...
Loading customers...
Loading articles...
Transactions: (31788324, 5)
Customers:    (1371980, 7)
Articles:     (105542, 25)


## Cell 3 — Inspect Data

In [3]:
print('=== TRANSACTIONS ===')
print(transactions.head(3))
print(transactions.dtypes)
print(transactions.isnull().sum())

print('\n=== CUSTOMERS ===')
print(customers.head(3))
print(customers.isnull().sum())

print('\n=== ARTICLES ===')
print(articles.head(3))
print(articles.columns.tolist())

=== TRANSACTIONS ===
       t_dat                                        customer_id  article_id  \
0 2018-09-20  000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...  0663713001   
1 2018-09-20  000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...  0541518023   
2 2018-09-20  00007d2de826758b65a93dd24ce629ed66842531df6699...  0505221004   

      price  sales_channel_id  
0  0.050831                 2  
1  0.030492                 2  
2  0.015237                 2  
t_dat               datetime64[ns]
customer_id                 object
article_id                  object
price                      float64
sales_channel_id             int64
dtype: object
t_dat               0
customer_id         0
article_id          0
price               0
sales_channel_id    0
dtype: int64

=== CUSTOMERS ===
                                         customer_id  FN  Active  \
0  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3... NaN     NaN   
1  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e... NaN     NaN   

## Cell 4 — Clean Customers

In [4]:
# Drop postal_code - too granular, adds noise
customers = customers.drop(columns=['postal_code'])

# Fill missing age with median
customers['age'] = customers['age'].fillna(customers['age'].median())

# Fill missing categorical columns
customers['club_member_status'] = customers['club_member_status'].fillna('UNKNOWN')
customers['fashion_news_frequency'] = customers['fashion_news_frequency'].fillna('NONE')

# Binary encode club membership
customers['is_club_member'] = (customers['club_member_status'] == 'ACTIVE').astype(int)

# Ordinal encode fashion news frequency
news_map = {'NONE': 0, 'Monthly': 1, 'Regularly': 2, 'UNKNOWN': 0}
customers['news_frequency_encoded'] = customers['fashion_news_frequency'].map(news_map).fillna(0)

print(customers.head())
print(f'Customers shape: {customers.shape}')

                                         customer_id   FN  Active  \
0  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...  NaN     NaN   
1  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...  NaN     NaN   
2  000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...  NaN     NaN   
3  00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...  NaN     NaN   
4  00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...  1.0     1.0   

  club_member_status fashion_news_frequency   age  is_club_member  \
0             ACTIVE                   NONE  49.0               1   
1             ACTIVE                   NONE  25.0               1   
2             ACTIVE                   NONE  24.0               1   
3             ACTIVE                   NONE  54.0               1   
4             ACTIVE              Regularly  52.0               1   

   news_frequency_encoded  
0                       0  
1                       0  
2                       0  
3                       0  
4                       2  
Cu

## Cell 5 — Clean Articles (keep only useful columns)

In [5]:
articles_slim = articles[[
    'article_id',
    'product_type_name',
    'department_name',
    'section_name',
    'colour_group_name',
    'perceived_colour_value_name'
]].copy()

print(articles_slim.head())
print(articles_slim.isnull().sum())

   article_id product_type_name department_name            section_name  \
0  0108775015          Vest top    Jersey Basic  Womens Everyday Basics   
1  0108775044          Vest top    Jersey Basic  Womens Everyday Basics   
2  0108775051          Vest top    Jersey Basic  Womens Everyday Basics   
3  0110065001               Bra  Clean Lingerie         Womens Lingerie   
4  0110065002               Bra  Clean Lingerie         Womens Lingerie   

  colour_group_name perceived_colour_value_name  
0             Black                        Dark  
1             White                       Light  
2         Off White                 Dusty Light  
3             Black                        Dark  
4             White                       Light  
article_id                     0
product_type_name              0
department_name                0
section_name                   0
colour_group_name              0
perceived_colour_value_name    0
dtype: int64


## Cell 6 — Merge Transactions with Articles

In [6]:
df = transactions.merge(articles_slim, on='article_id', how='left')
print(f'Merged dataframe shape: {df.shape}')
print(df.head(3))

Merged dataframe shape: (31788324, 10)
       t_dat                                        customer_id  article_id  \
0 2018-09-20  000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...  0663713001   
1 2018-09-20  000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...  0541518023   
2 2018-09-20  00007d2de826758b65a93dd24ce629ed66842531df6699...  0505221004   

      price  sales_channel_id product_type_name      department_name  \
0  0.050831                 2    Underwear body  Expressive Lingerie   
1  0.030492                 2               Bra      Casual Lingerie   
2  0.015237                 2           Sweater     Tops Knitwear DS   

       section_name colour_group_name perceived_colour_value_name  
0   Womens Lingerie             Black                        Dark  
1   Womens Lingerie        Light Pink                 Dusty Light  
2  Divided Selected              Pink                Medium Dusty  


## Cell 7 — Layer 1: RFM Features

In [7]:
snapshot_date = df['t_dat'].max() + pd.Timedelta(days=1)

rfm = df.groupby('customer_id').agg(
    recency_days    = ('t_dat', lambda x: (snapshot_date - x.max()).days),
    frequency       = ('t_dat', 'count'),
    monetary_total  = ('price', 'sum'),
    avg_basket_size = ('price', 'mean'),
    monetary_median = ('price', 'median')
).reset_index()

print(rfm.describe())

       recency_days     frequency  monetary_total  avg_basket_size  \
count  1.362281e+06  1.362281e+06    1.362281e+06     1.362281e+06   
mean   2.361484e+02  2.333463e+01    6.493858e-01     2.875476e-02   
std    2.211188e+02  3.924225e+01    1.197582e+00     1.429338e-02   
min    1.000000e+00  1.000000e+00    7.627119e-04     7.627119e-04   
25%    4.900000e+01  3.000000e+00    8.806780e-02     2.095763e-02   
50%    1.520000e+02  9.000000e+00    2.455932e-01     2.613317e-02   
75%    3.990000e+02  2.700000e+01    7.011864e-01     3.336083e-02   
max    7.340000e+02  1.895000e+03    5.767641e+01     5.067797e-01   

       monetary_median  
count     1.362281e+06  
mean      2.682891e-02  
std       1.455704e-02  
min       5.593220e-04  
25%       1.693220e-02  
50%       2.540678e-02  
75%       3.218644e-02  
max       5.067797e-01  


## Cell 8 — Layer 2: Temporal Features
> ⚠️ This cell may take 5–10 minutes due to row-wise computation on 31M rows. Please be patient.
>
> **Key improvement:** `purchase_trend_slope` is now normalized to [-1, +1] using
> `(second - first) / (second + first + 1)` so it is scale-independent across
> customers with very different purchase volumes.

In [8]:
# Sort once
df_sorted = df[['customer_id', 't_dat']].sort_values(['customer_id', 't_dat'])

# Avg inter-purchase gap — vectorized using diff()
df_sorted['gap'] = df_sorted.groupby('customer_id')['t_dat'].diff().dt.days
avg_gap = df_sorted.groupby('customer_id')['gap'].mean().reset_index()
avg_gap.columns = ['customer_id', 'avg_inter_purchase_gap']

# Active months — number of distinct calendar months with at least one purchase
active_months = df.groupby('customer_id')['t_dat'].apply(
    lambda x: x.dt.to_period('M').nunique()
).reset_index()
active_months.columns = ['customer_id', 'active_months']

# Dominant quarter — which quarter of the year does this customer shop most?
df['quarter'] = df['t_dat'].dt.quarter
dominant_quarter = df.groupby('customer_id')['quarter'].agg(
    lambda x: x.mode()[0]
).reset_index()
dominant_quarter.columns = ['customer_id', 'dominant_quarter']

# Purchase trend slope — NORMALIZED to [-1, +1]
# Positive = customer buying more in second half of their history (growing)
# Negative = customer buying less in second half (declining)
# Formula: (second_half - first_half) / (second_half + first_half + 1)
# The +1 prevents division by zero for single-transaction customers.
# This makes the metric scale-independent: a customer with 4 purchases
# is fairly compared with one who has 400.
df_sorted['rank'] = df_sorted.groupby('customer_id').cumcount()
half_mark = df_sorted.groupby('customer_id')['rank'].transform('max') // 2
first_half = df_sorted[df_sorted['rank'] <= half_mark]
second_half = df_sorted[df_sorted['rank'] > half_mark]

first_counts  = first_half.groupby('customer_id').size().reset_index(name='first_half_count')
second_counts = second_half.groupby('customer_id').size().reset_index(name='second_half_count')
trend = first_counts.merge(second_counts, on='customer_id', how='outer').fillna(0)

trend['purchase_trend_slope'] = (
    (trend['second_half_count'] - trend['first_half_count']) /
    (trend['second_half_count'] + trend['first_half_count'] + 1)
)
trend = trend[['customer_id', 'purchase_trend_slope']]

# Merge all temporal features
temporal = avg_gap.merge(active_months, on='customer_id', how='left')
temporal = temporal.merge(dominant_quarter, on='customer_id', how='left')
temporal = temporal.merge(trend, on='customer_id', how='left')

# Fill NaN gaps for single-purchase customers
temporal['avg_inter_purchase_gap'] = temporal['avg_inter_purchase_gap'].fillna(
    temporal['avg_inter_purchase_gap'].median()
)

print(temporal.describe())
print(f'\npurchase_trend_slope range: [{temporal["purchase_trend_slope"].min():.3f}, {temporal["purchase_trend_slope"].max():.3f}]')
print('(should be between -1 and +1)')

       avg_inter_purchase_gap  active_months  dominant_quarter  \
count            1.362281e+06   1.362281e+06      1.362281e+06   
mean             2.067515e+01   4.573462e+00      2.492989e+00   
std              3.707991e+01   4.725334e+00      1.063186e+00   
min              0.000000e+00   1.000000e+00      1.000000e+00   
25%              1.000000e+00   1.000000e+00      2.000000e+00   
50%              1.105455e+01   2.000000e+00      2.000000e+00   
75%              2.342857e+01   6.000000e+00      3.000000e+00   
max              7.290000e+02   2.500000e+01      4.000000e+00   

       purchase_trend_slope  
count          1.362281e+06  
mean          -9.023038e-02  
std            1.517267e-01  
min           -5.000000e-01  
25%           -1.000000e-01  
50%           -9.090909e-03  
75%            0.000000e+00  
max            0.000000e+00  

purchase_trend_slope range: [-0.500, 0.000]
(should be between -1 and +1)


## Cell 9 — Layer 3: Product Affinity Features

In [9]:
# Top department per customer
top_dept = df.groupby('customer_id')['department_name'].agg(
    lambda x: x.mode()[0] if len(x) > 0 else 'Unknown'
).reset_index().rename(columns={'department_name': 'top_department'})

# Category diversity — unique product types purchased
cat_diversity = df.groupby('customer_id')['product_type_name'].nunique().reset_index()
cat_diversity.columns = ['customer_id', 'category_diversity_score']

# Average price tier
price_tier = df.groupby('customer_id')['price'].mean().reset_index()
price_tier.columns = ['customer_id', 'avg_price_tier']

# Price variability — std of prices paid; captures whether customer is
# price-sensitive (low std = consistent budget) or exploratory (high std)
price_std = df.groupby('customer_id')['price'].std().reset_index()
price_std.columns = ['customer_id', 'price_std']
price_std['price_std'] = price_std['price_std'].fillna(0)  # single-purchase customers

# Top colour group
top_colour = df.groupby('customer_id')['colour_group_name'].agg(
    lambda x: x.mode()[0] if len(x) > 0 else 'Unknown'
).reset_index().rename(columns={'colour_group_name': 'top_colour'})

# Top section
top_section = df.groupby('customer_id')['section_name'].agg(
    lambda x: x.mode()[0] if len(x) > 0 else 'Unknown'
).reset_index().rename(columns={'section_name': 'top_section'})

# Merge all product affinity features
product_affinity = top_dept.merge(cat_diversity, on='customer_id')
product_affinity = product_affinity.merge(price_tier, on='customer_id')
product_affinity = product_affinity.merge(price_std, on='customer_id')
product_affinity = product_affinity.merge(top_colour, on='customer_id')
product_affinity = product_affinity.merge(top_section, on='customer_id')

print(product_affinity.head())
print(f'Product affinity shape: {product_affinity.shape}')

                                         customer_id       top_department  \
0  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...                 Suit   
1  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...             Swimwear   
2  000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...               Blouse   
3  00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...    Ladies Sport Bras   
4  00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...  Functional Lingerie   

   category_diversity_score  avg_price_tier  price_std top_colour  \
0                        13        0.030904   0.015718      Black   
1                        23        0.030255   0.016957      Black   
2                        10        0.039154   0.016858      Black   
3                         1        0.030492   0.000000      Black   
4                         9        0.036130   0.012638      Black   

                  top_section  
0            Womens Tailoring  
1  Womens Swimwear, beachwear  
2               Womens Cas

## Cell 10 — Layer 4: Loyalty & Repeat Purchase Features
> **New layer added in v2.**
>
> `repeat_purchase_rate` captures how often a customer re-buys the *same* article.
> A high rate signals brand loyalty and habitual buying; a low rate signals
> exploratory behavior. This is a fashion-specific signal not captured by RFM alone.

In [10]:
# Repeat purchase rate — proportion of transactions that are re-buys of the same article
# Value of 0 = customer never buys the same item twice (explorer)
# Value of 1 = customer exclusively re-buys items they've bought before (loyalist)
repeat_rate = (
    df.groupby('customer_id')['article_id']
    .apply(lambda x: x.duplicated().sum() / len(x))
    .reset_index()
)
repeat_rate.columns = ['customer_id', 'repeat_purchase_rate']

# Unique articles purchased — breadth of product exploration
unique_articles = df.groupby('customer_id')['article_id'].nunique().reset_index()
unique_articles.columns = ['customer_id', 'unique_articles_count']

loyalty_features = repeat_rate.merge(unique_articles, on='customer_id')

print(loyalty_features.describe())
print(f'\nSample repeat_purchase_rate distribution:')
print(pd.cut(loyalty_features['repeat_purchase_rate'], bins=5).value_counts().sort_index())

       repeat_purchase_rate  unique_articles_count
count          1.362281e+06           1.362281e+06
mean           9.514399e-02           2.004464e+01
std            1.433676e-01           3.198025e+01
min            0.000000e+00           1.000000e+00
25%            0.000000e+00           3.000000e+00
50%            0.000000e+00           8.000000e+00
75%            1.428571e-01           2.400000e+01
max            9.900000e-01           1.346000e+03

Sample repeat_purchase_rate distribution:
repeat_purchase_rate
(-0.00099, 0.198]    1107004
(0.198, 0.396]        178273
(0.396, 0.594]         62529
(0.594, 0.792]         11973
(0.792, 0.99]           2502
Name: count, dtype: int64


## Cell 11 — Merge All Feature Layers

In [11]:
# Merge all four layers
features = rfm.merge(temporal, on='customer_id', how='left')
features = features.merge(product_affinity, on='customer_id', how='left')
features = features.merge(loyalty_features, on='customer_id', how='left')

# Add demographics
demo_cols = ['customer_id', 'age', 'is_club_member', 'news_frequency_encoded']
features = features.merge(customers[demo_cols], on='customer_id', how='left')

print(f'Final feature table shape: {features.shape}')
print(f'\nFeature columns:')
for col in features.columns:
    print(f'  {col}')
print('\nNull counts:')
print(features.isnull().sum())

Final feature table shape: (1362281, 21)

Feature columns:
  customer_id
  recency_days
  frequency
  monetary_total
  avg_basket_size
  monetary_median
  avg_inter_purchase_gap
  active_months
  dominant_quarter
  purchase_trend_slope
  top_department
  category_diversity_score
  avg_price_tier
  price_std
  top_colour
  top_section
  repeat_purchase_rate
  unique_articles_count
  age
  is_club_member
  news_frequency_encoded

Null counts:
customer_id                 0
recency_days                0
frequency                   0
monetary_total              0
avg_basket_size             0
monetary_median             0
avg_inter_purchase_gap      0
active_months               0
dominant_quarter            0
purchase_trend_slope        0
top_department              0
category_diversity_score    0
avg_price_tier              0
price_std                   0
top_colour                  0
top_section                 0
repeat_purchase_rate        0
unique_articles_count       0
age            

## Cell 12 — Encode Categorical Columns with Frequency Encoding
> **Changed from v1:** `LabelEncoder` has been replaced with **frequency encoding**.
>
> **Why this matters:** `LabelEncoder` assigns arbitrary integers (e.g. Divided=2, Ladies=5)
> which implies a false numeric ordering. K-Means interprets 5 as "more than" 2, which is
> meaningless for department names. Frequency encoding replaces each category with its
> proportion among all customers, producing a legitimate continuous signal that reflects
> how common each preference is across the customer base.

In [12]:
cat_cols = ['top_department', 'top_colour', 'top_section']

for col in cat_cols:
    # Compute frequency of each category across all customers
    freq_map = features[col].value_counts(normalize=True)
    features[col + '_enc'] = features[col].map(freq_map).fillna(0)
    print(f'{col}: {features[col].nunique()} unique categories → encoded as frequency')

# dominant_quarter is already a legitimate ordinal integer (1,2,3,4) — keep as-is
print('\ndominant_quarter kept as integer (1=Q1, 2=Q2, 3=Q3, 4=Q4)')

# Drop original string columns, keep encoded versions
features_model = features.drop(columns=cat_cols)

print(f'\nFinal feature matrix shape: {features_model.shape}')
print(features_model.dtypes)

top_department: 242 unique categories → encoded as frequency
top_colour: 50 unique categories → encoded as frequency
top_section: 56 unique categories → encoded as frequency

dominant_quarter kept as integer (1=Q1, 2=Q2, 3=Q3, 4=Q4)

Final feature matrix shape: (1362281, 21)
customer_id                  object
recency_days                  int64
frequency                     int64
monetary_total              float64
avg_basket_size             float64
monetary_median             float64
avg_inter_purchase_gap      float64
active_months                 int64
dominant_quarter              int32
purchase_trend_slope        float64
category_diversity_score      int64
avg_price_tier              float64
price_std                   float64
repeat_purchase_rate        float64
unique_articles_count         int64
age                         float64
is_club_member                int64
news_frequency_encoded        int64
top_department_enc          float64
top_colour_enc              float64
top_

## Cell 13 — Final Check & Save to Drive

In [13]:
# Drop any remaining nulls
features_model = features_model.dropna()
print(f'Final shape after dropping nulls: {features_model.shape}')
print(features_model.describe())

# Quick sanity checks
assert features_model['purchase_trend_slope'].between(-1, 1).all(), \
    'ERROR: purchase_trend_slope out of [-1, 1] range'
assert features_model['repeat_purchase_rate'].between(0, 1).all(), \
    'ERROR: repeat_purchase_rate out of [0, 1] range'
print('\nSanity checks passed.')

# Save to Drive
OUTPUT = '/content/drive/MyDrive/HM_Segmentation/data/'
features_model.to_csv(OUTPUT + 'HM_features.csv', index=False)
print('\nHM_features.csv saved to Drive successfully!')
print(f'Total customers: {len(features_model):,}')
print(f'Total features: {features_model.shape[1] - 1} (excluding customer_id)')

Final shape after dropping nulls: (1362281, 21)
       recency_days     frequency  monetary_total  avg_basket_size  \
count  1.362281e+06  1.362281e+06    1.362281e+06     1.362281e+06   
mean   2.361484e+02  2.333463e+01    6.493858e-01     2.875476e-02   
std    2.211188e+02  3.924225e+01    1.197582e+00     1.429338e-02   
min    1.000000e+00  1.000000e+00    7.627119e-04     7.627119e-04   
25%    4.900000e+01  3.000000e+00    8.806780e-02     2.095763e-02   
50%    1.520000e+02  9.000000e+00    2.455932e-01     2.613317e-02   
75%    3.990000e+02  2.700000e+01    7.011864e-01     3.336083e-02   
max    7.340000e+02  1.895000e+03    5.767641e+01     5.067797e-01   

       monetary_median  avg_inter_purchase_gap  active_months  \
count     1.362281e+06            1.362281e+06   1.362281e+06   
mean      2.682891e-02            2.067515e+01   4.573462e+00   
std       1.455704e-02            3.707991e+01   4.725334e+00   
min       5.593220e-04            0.000000e+00   1.000000e+00